### 时序差分算法

#### 简介
- 无模型强化学习 (model-free reinforcement learning): 智能体只能和环境进行交互，通过采样到的数据来学习
- model-free 经典算法: 时序差分 (temporal difference，TD)、Sarsa、Q-Learning
- 一组概念: 在线策略学习、离线策略学习
- 在线策略学习: 要求使用在当前策略下采样得到的样本进行学习，一旦策略被更新，当前的样本就被放弃了
- 离线策略学习: 使用经验回放池将之前采样得到的样本收集起来再次利用; 更好地利用历史数据，具有更小的样本复杂度

#### 时序差分算法
- 时序差分是一种用来估计一个策略的价值函数的方法，它结合了蒙特卡洛和动态规划算法的思想
- 蒙特卡洛结合: 可以从样本数据中学习，不需要事先知道环境
- 动态规划结合: 根据贝尔曼方程的思想，利用后续状态的价值估计来更新当前状态的价值估计

1. TD Learning (estimate state value) 定义: 给定策略, 利用当前一步获得的 Reward 和下一状态的 Value estimate 来更新当前状态的 Value

- 核心公式:

$$
v_{t+1}(s_t)
=
v_t(s_t)
-
\alpha_t(s_t)
\left[
v_t(s_t)
-
\left(
r_{t+1}
+
\gamma v_t(s_{t+1})
\right)
\right]
$$

$$
v_{t+1}(s)
=
v_t(s),
\qquad
\forall s \neq s_t
$$

2. TD target 定义: TD Target 是“即时奖励 + 下一状态估计价值”，作为当前状态应该逼近的目标

- 核心公式:

$$
\bar{v}_t
=
r_{t+1}
+
\gamma v_t(s_{t+1})
$$

3. TD error 定义: TD Error 衡量当前 Value estimate 与 TD Target 之间的误差

- 核心公式: 

$$
\delta_t
=
v_t(s_t)
-
\left[
r_{t+1}
+
\gamma v_t(s_{t+1})
\right]
$$

#### Sarsa 算法 （State–Action–Reward–State–Action）
- 定义: 通过与环境交互，利用当前状态、动作、奖励以及下一状态和动作，不断更新动作价值函数 $Q(s,a)$
- 核心公式: 

$$
Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha \left[r_t + \gamma Q(s_{t+1},a_{t+1}) - Q(s_t,a_t)\right]
$$

- Sarsa = ε-Greedy 选择动作 + TD 更新动作价值 Q + 不断改进策略




In [ ]:
import numpy as np

class Sarsa:
    """ Sarsa算法 """
    def __init__(self, ncol, nrow, epsilon, alpha, gamma, n_action=4):
        self.Q_table = np.zeros([nrow * ncol, n_action])  # 初始化Q(s,a)表格
        self.n_action = n_action   # 动作个数
        self.alpha = alpha         # 学习率
        self.gamma = gamma         # 折扣因子
        self.epsilon = epsilon     # epsilon-贪婪策略中的参数

    def take_action(self, state):  # 根据ε-Greedy策略，在当前状态选择动作
        if np.random.random() < self.epsilon:          # 以 ε 的概率进行随机探索
            action = np.random.randint(self.n_action)  # 从所有动作中随机选择一个
        else:                                          # 以 1-ε 的概率进行利用
            action = np.argmax(self.Q_table[state])    # 选择当前状态下Q值最大的动作
        return action

    def best_action(self, state):  # 找出当前状态下所有Q值最大的动作，用于打印策略
        Q_max = np.max(self.Q_table[state])            # 获取当前状态下最大的Q值
        a = [0 for _ in range(self.n_action)]          # 创建长度为动作数量的全0列表
        for i in range(self.n_action):                 # 若两个动作的价值一样,都会记录下来
            if self.Q_table[state, i] == Q_max:        # 判断当前动作的Q值是否等于最大值  
                a[i] = 1                               # 将所有具有最大Q值的动作标记为1
        return a

    def update(self, s0, a0, r, s1, a1):               # 根据当前和下一状态动作信息更新Q值
        td_error = r + self.gamma * self.Q_table[s1, a1] - self.Q_table[s0, a0] # 计算TD误差
        self.Q_table[s0, a0] += self.alpha * td_error  # 利用TD误差更新当前状态动作对的Q值
        

#### 多步 Sarsa 算法
- 定义: 通过累积未来 $n$ 步的实际奖励，并结合第 $n$ 步后的动作价值估计，更新当前动作价值函数 $Q(s,a)$
- 核心公式: 

$$
Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha \left[ r_t + \gamma r_{t+1} + \cdots + \gamma^n Q(s_{t+n},a_{t+n}) - Q(s_t,a_t) \right]
$$

- 多步 Sarsa 通过控制真实奖励的采样步数 $n$，在 TD 方法的低方差与 MC 方法的低自举偏差之间取得平衡，从而有机会提升学习效率和稳定性

In [ ]:
import numpy as np

class nstep_Sarsa:
    """n步Sarsa算法"""

    def __init__(self, n, ncol, nrow, epsilon, alpha, gamma, n_action=4):  # 初始化n步Sarsa智能体
        self.Q_table = np.zeros([nrow * ncol, n_action])  # 创建状态数×动作数的全0 Q表
        self.n_action = n_action   # 保存动作数量
        self.alpha = alpha         # 设置学习率
        self.gamma = gamma         # 设置折扣因子
        self.epsilon = epsilon     # 设置ε-Greedy的探索概率
        self.n = n                 # 设置多步Sarsa的步数n
        self.state_list = []       # 缓存最近的状态
        self.action_list = []      # 缓存最近的动作
        self.reward_list = []      # 缓存最近的奖励

    def take_action(self, state):                      # 使用ε-Greedy策略选择动作
        if np.random.random() < self.epsilon:          # 以ε的概率随机探索
            action = np.random.randint(self.n_action)  # 随机选择一个动作
        else:                                          # 以1-ε的概率利用当前Q值
            action = np.argmax(self.Q_table[state])    # 选择当前状态下Q值最大的动作
        return action                                  # 返回选中的动作

    def best_action(self, state):                # 获取当前状态下的所有最优动作
        Q_max = np.max(self.Q_table[state])      # 找出当前状态下最大的Q值
        a = [0 for _ in range(self.n_action)]    # 初始化动作标记列表
        for i in range(self.n_action):           # 遍历所有动作
            if self.Q_table[state, i] == Q_max:  # 判断当前动作是否具有最大Q值
                a[i] = 1                         # 将最优动作标记为1
        return a                                 # 返回最优动作标记列表

    def update(self, s0, a0, r, s1, a1, done):  # 接收一次交互数据并执行多步更新
        self.state_list.append(s0)              # 保存当前状态s0
        self.action_list.append(a0)             # 保存当前动作a0
        self.reward_list.append(r)              # 保存本次获得的奖励r

        if len(self.state_list) == self.n:      # 当缓存积累到n步时开始更新
            G = self.Q_table[s1, a1]            # 初始化n步回报的末端Q值

            for i in reversed(range(self.n)):   # 从第n步向第1步逆序计算回报
                G = self.gamma * G + self.reward_list[i]  # 递推计算折扣回报G

                if done and i > 0:              # 回合终止时额外更新缓存中后面几步的Q值
                    s = self.state_list[i]      # 获取对应的历史状态
                    a = self.action_list[i]     # 获取对应的历史动作
                    self.Q_table[s, a] += self.alpha * (G - self.Q_table[s, a])  # 使用剩余回报更新Q值

            s = self.state_list.pop(0)          # 取出并删除缓存中最早的状态
            a = self.action_list.pop(0)         # 取出并删除缓存中最早的动作
            self.reward_list.pop(0)             # 删除缓存中最早的奖励

            self.Q_table[s, a] += self.alpha * (G - self.Q_table[s, a])  # 使用n步回报更新最早的状态动作Q值

        if done:  # 如果当前回合结束
            self.state_list = []   # 清空状态缓存
            self.action_list = []  # 清空动作缓存
            self.reward_list = []  # 清空奖励缓存

#### Q-Learning 
- 定义: 通过不断更新动作价值函数 $Q(s,a)$, 学习最优动作价值函数 $Q^*(s,a)$, 进而得到最优策略
- 核心公式: 

$$
Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha \left[r_t + \gamma \max_a Q(s_{t+1},a) - Q(s_t,a_t)\right]
$$

In [ ]:
class QLearning:
    """Q-learning算法"""

    def __init__(self, ncol, nrow, epsilon, alpha, gamma, n_action=4):  # 初始化Q-learning智能体
        self.Q_table = np.zeros([nrow * ncol, n_action])  # 创建状态数×动作数的全0 Q表
        self.n_action = n_action                          # 保存动作数量，默认为4
        self.alpha = alpha                                # 设置学习率，控制Q值更新幅度
        self.gamma = gamma                                # 设置折扣因子，控制未来奖励权重
        self.epsilon = epsilon                            # 设置ε-Greedy策略的随机探索概率

    def take_action(self, state):                         # 根据ε-Greedy策略选择当前动作
        if np.random.random() < self.epsilon:             # 以ε的概率进行随机探索
            action = np.random.randint(self.n_action)     # 从所有动作中随机选择一个
        else:                                             # 以1-ε的概率利用当前Q值
            action = np.argmax(self.Q_table[state])       # 选择当前状态下Q值最大的动作
        return action                                     # 返回选中的动作编号

    def best_action(self, state):                  # 获取当前状态下所有Q值最大的动作，用于打印策略
        Q_max = np.max(self.Q_table[state])        # 获取当前状态下最大的Q值
        a = [0 for _ in range(self.n_action)]      # 创建长度为动作数量的全0列表
        for i in range(self.n_action):             # 遍历所有动作
            if self.Q_table[state, i] == Q_max:    # 判断当前动作的Q值是否等于最大值
                a[i] = 1                           # 将具有最大Q值的动作标记为1
        return a                                   # 返回最优动作标记列表

    def update(self, s0, a0, r, s1):  # 根据当前交互数据更新Q值
        td_error = r + self.gamma * self.Q_table[s1].max() - self.Q_table[s0, a0]  # 计算TD误差
        self.Q_table[s0, a0] += self.alpha * td_error  # 利用TD误差更新当前状态动作对的Q值

#### 在线策略算法和离线策略算法
- 行为策略 (behavior policy): 采样数据的策略
- 目标策略 (target policy): 用采样的数据来更新的策略
- 同策略 (on-policy): 表示行为策略和目标策略是同一个策略
- 异策略 (off-policy): 表示行为策略和目标策略不是同一个策略
- 在线强化学习 (Online RL):	训练过程中持续与环境交互，收集新数据
- 离线强化学习 (Offline RL): 使用固定的历史数据集训练，不再与环境交互采集新数据
